# Tomahawk TF48 — independent native XGRID worker Part 3

Run Parts 1, 2, and 3 simultaneously. This worker validates the 16 T / 48 physical-coil / 16-field-period package, computes the direct under-1% ripple preflight, and runs only its own native XGRID chunk. It does not wait for another worker. Use the assembler after all three output ZIPs are collected. Attach the corrected handoff ZIP and your existing `x-grid-zip` runtime dataset; the notebook discovers and stages its `xgrid` executable and libraries automatically. The native subprocess has no internal timeout by default, prints a heartbeat every 60 seconds, and only completes green after a valid native partial MGRID is created.

In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from pathlib import Path

try:
    import numpy as np
    import scipy
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "numpy", "scipy"])
    import numpy as np
    import scipy

PART_INDEX = 3
COIL_FILE = f"coils.tomahawk_tf48_part{PART_INDEX}"
INPUT_FILE = f"input_xgrid_tomahawk_tf48_part{PART_INDEX}"
EXPECTED_MGRID = f"mgrid_tomahawk_tf48_part{PART_INDEX}.nc"
TARGET_CURRENT_A = -125000.0
TARGET_AXIS_FIELD_T = 16.0
RIPPLE_LIMIT_PERCENT = 1.0
EXPECTED_RIPPLE_PERCENT = 0.7539178029174242
REQUIRED_DIMS = {"phi": 36, "zee": 201, "rad": 201}
FIELD_VARS = ("br_001", "bp_001", "bz_001")


class WorkerFailure(RuntimeError):
    """A fail-closed worker result that must make the notebook cell fail."""


def nonnegative_env_seconds(name: str, default: float) -> float:
    raw = os.environ.get(name, str(default)).strip()
    value = float(raw)
    if not math.isfinite(value) or value < 0.0:
        raise ValueError(f"{name} must be a finite non-negative number")
    return value


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def safe_extract(archive: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with zipfile.ZipFile(archive) as zf:
        for info in zf.infolist():
            target = (destination / info.filename).resolve()
            if target != base and base not in target.parents:
                raise RuntimeError(f"unsafe archive member: {info.filename}")
        zf.extractall(destination)


def search_roots(work: Path) -> list[Path]:
    values = []
    if os.environ.get("TF48_DATA_ROOT"):
        values.append(Path(os.environ["TF48_DATA_ROOT"]).expanduser())
    here = Path.cwd().resolve()
    values.extend([here, here.parent, here.parent.parent, here.parent.parent.parent, work, work.parent])
    values.extend(Path(item) for item in ("/kaggle/input", "/kaggle/working", "/kaggle/models", "/data", "/mnt/data"))
    result = []
    seen = set()
    for value in values:
        value = value.resolve()
        if value.exists() and value not in seen:
            result.append(value)
            seen.add(value)
    return result


def package_archives(roots: list[Path]) -> list[Path]:
    result = []
    seen = set()
    for root in roots:
        if root.is_file() and root.suffix.lower() == ".zip":
            items = [root]
        elif root.is_dir():
            try:
                items = list(root.rglob("*.zip"))
            except OSError:
                items = []
        else:
            items = []
        for item in items:
            if item not in seen and any(token in item.name.lower() for token in ("tf48", "tomahawk", "x-grid", "xgrid", "makegrid", "stellopt")):
                result.append(item)
                seen.add(item)
    return result


def validate_root(root: Path) -> tuple[bool, list[str]]:
    errors = []
    contract_path = root / "tf48_16t_input_contract.json"
    manifest_path = root / "dataset_manifest.json"
    if not contract_path.is_file() or not manifest_path.is_file():
        return False, ["contract or dataset_manifest missing"]
    try:
        contract = json.loads(contract_path.read_text())
        manifest = json.loads(manifest_path.read_text())
    except Exception as exc:
        return False, [f"invalid JSON: {exc}"]
    config = contract.get("configuration", {})
    if config.get("target_axis_field_T") != 16.0:
        errors.append("target field is not 16 T")
    if config.get("physical_coils") != 48:
        errors.append("physical coil count is not 48")
    if config.get("field_periods") != 16:
        errors.append("field periods is not 16")
    if contract.get("current_schedule", {}).get("canonical_current_A") != TARGET_CURRENT_A:
        errors.append("canonical current is not -125000 A/turn")
    for entry in manifest.get("files", []):
        path = root / entry["path"]
        if not path.is_file():
            errors.append(f"missing {entry['path']}")
        elif sha256(path) != entry.get("sha256"):
            errors.append(f"SHA-256 mismatch {entry['path']}")
    declared = manifest.get("contract_sha256")
    if declared and declared != sha256(contract_path):
        errors.append("contract SHA-256 mismatch")
    return not errors, errors


def find_data_root(work: Path) -> tuple[Path | None, list[str]]:
    roots = search_roots(work)
    candidates = []
    for root in roots:
        if root.is_dir():
            try:
                candidates.extend(path.parent for path in root.rglob("tf48_16t_input_contract.json"))
            except OSError:
                pass
    extracted = work / "input_archives"
    for archive in package_archives(roots):
        destination = extracted / archive.stem
        marker = destination / ".extracted"
        if not marker.exists():
            try:
                safe_extract(archive, destination)
                marker.write_text("ok\n")
            except Exception:
                continue
        try:
            candidates.extend(path.parent for path in destination.rglob("tf48_16t_input_contract.json"))
        except OSError:
            pass
    errors = []
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        valid, candidate_errors = validate_root(candidate)
        if valid:
            return candidate, []
        errors.extend([f"{candidate}: {error}" for error in candidate_errors])
    return None, errors


def compute_direct_ripple(data_root: Path) -> dict:
    rows = []
    path = data_root / "cad_a2_tf48_interleaved_centerlines.csv"
    with path.open(newline="") as f:
        for row in csv.DictReader(f):
            rows.append([float(row[key]) for key in ("x_m", "y_m", "z_m", "next_x_m", "next_y_m", "next_z_m")])
    points = np.asarray(rows, dtype=float)
    p1, p2 = points[:, :3], points[:, 3:]
    current = TARGET_CURRENT_A * 100.0
    mu0 = 4.0e-7 * math.pi
    bphi = []
    for phi in np.linspace(0.0, 2.0 * math.pi / 48.0, 128):
        q = np.array([10.5 * math.cos(phi), 10.5 * math.sin(phi), 0.0])
        a, b = q - p1, q - p2
        amag, bmag = np.linalg.norm(a, axis=1), np.linalg.norm(b, axis=1)
        cross_ab = np.cross(a, b)
        denom = amag * bmag * (amag * bmag + np.sum(a * b, axis=1))
        denom = np.where(denom == 0.0, np.inf, denom)
        factor = (amag + bmag) / denom
        field = (mu0 * current / (4.0 * math.pi)) * (factor[:, None] * cross_ab).sum(axis=0)
        bphi.append(float(field @ np.array([-math.sin(phi), math.cos(phi), 0.0])))
    bphi = np.asarray(bphi)
    ripple = 100.0 * (float(bphi.max()) - float(bphi.min())) / (float(bphi.max()) + float(bphi.min()))
    return {
        "edge_R_m": 10.5,
        "edge_Z_m": 0.0,
        "samples": 128,
        "bphi_min_T": float(bphi.min()),
        "bphi_mean_T": float(bphi.mean()),
        "bphi_max_T": float(bphi.max()),
        "edge_ripple_percent": float(ripple),
        "limit_percent": RIPPLE_LIMIT_PERCENT,
        "strict_pass": bool(ripple < RIPPLE_LIMIT_PERCENT),
        "matches_contract_expectation": bool(abs(ripple - EXPECTED_RIPPLE_PERCENT) < 5.0e-5),
    }


def find_xgrid(search_roots: list[Path]) -> str | None:
    candidates = []
    if os.environ.get("STELLOPT_XGRID"):
        candidates.append(os.environ["STELLOPT_XGRID"])
    candidates.extend(["/home/STELLOPT/MAKEGRID/Release/xgrid", "/usr/local/bin/xgrid", shutil.which("xgrid") or ""])
    for root in search_roots:
        if root.is_dir():
            try:
                candidates.extend(str(path) for path in root.rglob("xgrid") if path.is_file())
            except OSError:
                pass
    for item in candidates:
        parts = shlex.split(item)
        if len(parts) == 1 and Path(parts[0]).is_file():
            return parts[0]
        if len(parts) == 1 and shutil.which(parts[0]):
            return shutil.which(parts[0])
    return None


def stage_xgrid(source: str, native_dir: Path) -> tuple[Path, dict]:
    """Stage an input-mounted XGRID binary and sibling lib directories."""
    source_path = Path(source)
    staged = native_dir / "xgrid"
    shutil.copy2(source_path, staged)
    staged.chmod(staged.stat().st_mode | 0o111)
    environment = os.environ.copy()
    library_paths = []
    for candidate in (source_path.parent / "lib", source_path.parent / "lib64"):
        if candidate.is_dir():
            target = native_dir / candidate.name
            shutil.copytree(candidate, target, dirs_exist_ok=True)
            library_paths.append(str(target))
    if library_paths:
        old = environment.get("LD_LIBRARY_PATH", "")
        environment["LD_LIBRARY_PATH"] = os.pathsep.join(library_paths + ([old] if old else []))
    return staged, environment


def run_native_xgrid(
    executable: Path,
    input_path: Path,
    native_dir: Path,
    log_path: Path,
    environment: dict,
) -> tuple[int, float, bool]:
    """Run XGRID with no internal time limit by default and emit heartbeats."""
    heartbeat_s = max(0.1, nonnegative_env_seconds("TF48_HEARTBEAT_SECONDS", 60.0))
    timeout_s = nonnegative_env_seconds("TF48_XGRID_TIMEOUT_SECONDS", 0.0)
    started = time.time()
    timeout_label = "UNLIMITED" if timeout_s == 0.0 else f"{timeout_s:.0f}s"
    print(f"TF48_PART{PART_INDEX}_NATIVE_XGRID_START timeout={timeout_label}", flush=True)
    with input_path.open("r") as stdin_file, log_path.open("w") as log_file:
        process = subprocess.Popen(
            [str(executable)],
            stdin=stdin_file,
            cwd=native_dir,
            text=True,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=environment,
        )
        while True:
            try:
                return_code = process.wait(timeout=heartbeat_s)
                elapsed = time.time() - started
                print(
                    f"TF48_PART{PART_INDEX}_NATIVE_XGRID_END "
                    f"return_code={return_code} elapsed_s={elapsed:.1f}",
                    flush=True,
                )
                return return_code, elapsed, False
            except subprocess.TimeoutExpired:
                elapsed = time.time() - started
                print(
                    f"TF48_PART{PART_INDEX}_NATIVE_XGRID_RUNNING "
                    f"elapsed_s={elapsed:.1f} pid={process.pid}",
                    flush=True,
                )
                if timeout_s > 0.0 and elapsed >= timeout_s:
                    process.terminate()
                    try:
                        return_code = process.wait(timeout=30.0)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        return_code = process.wait()
                    return return_code, time.time() - started, True


def print_log_tail(path: Path, lines: int = 40) -> None:
    if not path.is_file():
        return
    tail = path.read_text(errors="replace").splitlines()[-lines:]
    print(f"TF48_PART{PART_INDEX}_XGRID_LOG_TAIL_START", flush=True)
    for line in tail:
        print(line, flush=True)
    print(f"TF48_PART{PART_INDEX}_XGRID_LOG_TAIL_END", flush=True)


def inspect_mgrid(path: Path) -> dict:
    from scipy.io import netcdf_file
    errors = []
    try:
        with netcdf_file(str(path), "r", mmap=False) as nc:
            dimensions = {key: int(value) for key, value in nc.dimensions.items()}
            variables = {key: np.array(value.data).copy() for key, value in nc.variables.items()}
        for name, expected in REQUIRED_DIMS.items():
            if dimensions.get(name) != expected:
                errors.append(f"dimension {name}={dimensions.get(name)!r}, expected {expected}")
        field_abs_max = {}
        for name in FIELD_VARS:
            if name not in variables or variables[name].shape != (36, 201, 201):
                errors.append(f"missing or malformed {name}")
            elif not np.all(np.isfinite(variables[name])):
                errors.append(f"non-finite values in {name}")
            else:
                field_abs_max[name] = float(np.max(np.abs(variables[name])))
        if field_abs_max and max(field_abs_max.values()) == 0.0:
            errors.append("all native field arrays are identically zero")
        mode = "".join(np.asarray(variables.get("mgrid_mode", [])).astype("U1").reshape(-1).tolist())
        if mode != "S":
            errors.append(f"mgrid_mode={mode!r}, expected S")
        current_data = variables.get("raw_coil_cur", np.array([]))
        current = float(current_data.reshape(-1)[0]) if current_data.size else None
        if current is None or abs(current - TARGET_CURRENT_A) > 1.0e-3:
            errors.append(f"raw_coil_cur={current!r}, expected {TARGET_CURRENT_A}")
        axis = None
        if not errors and "ir" in variables and "jz" in variables:
            ir = int(round(float(np.asarray(variables["ir"]).reshape(-1)[0])))
            jz = int(round(float(np.asarray(variables["jz"]).reshape(-1)[0])))
            axis = float(variables["bp_001"][0, jz // 2, ir // 2] * current)
        return {"path": str(path), "sha256": sha256(path), "dimensions": dimensions, "mode": mode, "raw_coil_cur_A": current, "axis_field_T": axis, "field_abs_max_per_A": field_abs_max, "errors": errors, "valid": not errors}
    except Exception as exc:
        return {"path": str(path), "errors": [f"MGRID read failed: {exc}"], "valid": False}


def make_output_zip(work: Path, data_root: Path, report_path: Path, native_dir: Path | None) -> Path:
    output = work / f"TF48_16T_PART{PART_INDEX}_OUTPUT.zip"
    with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(report_path, report_path.name)
        for name in (COIL_FILE, INPUT_FILE, f"extcur.tomahawk_tf48_part{PART_INDEX}"):
            path = data_root / name
            if path.is_file():
                zf.write(path, name)
        if native_dir and native_dir.is_dir():
            for path in native_dir.iterdir():
                if path.is_file():
                    zf.write(path, f"native/{path.name}")
    return output


def main() -> None:
    started = time.time()
    if os.environ.get("TF48_WORKDIR"):
        work = Path(os.environ["TF48_WORKDIR"]).expanduser()
    elif Path("/kaggle/working").is_dir():
        work = Path("/kaggle/working") / f"tf48_part{PART_INDEX}_run"
    else:
        work = Path.cwd() / f"tf48_part{PART_INDEX}_run"
    work.mkdir(parents=True, exist_ok=True)
    report_path = work / f"TF48_16T_PART{PART_INDEX}_REPORT.json"
    report = {
        "schema_version": "TOMAHAWK-TF48-16T-PART-REPORT-2.0-HARD-FAIL",
        "part": PART_INDEX,
        "parallel_worker": True,
        "configuration": {"physical_coils": 48, "field_periods": 16, "target_axis_field_T": TARGET_AXIS_FIELD_T, "target_current_per_turn_A": TARGET_CURRENT_A},
        "status": "STARTED",
    }
    try:
        data_root, locate_errors = find_data_root(work)
        if data_root is None:
            report.update({"status": "INPUT_BLOCKED", "input_errors": locate_errors[-50:]})
            report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
            output = make_output_zip(work, work, report_path, None)
            print("INPUT_BLOCKED: no verified 16 T data root found")
            print(f"TF48_PART{PART_INDEX}_OUTPUT_ZIP={output}")
            raise WorkerFailure("INPUT_BLOCKED: no verified 16 T data root found")
        valid, errors = validate_root(data_root)
        report.update({"data_root": str(data_root), "input_manifest_pass": valid, "input_errors": errors})
        if not valid:
            report["status"] = "INPUT_BLOCKED"
            report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
            output = make_output_zip(work, data_root, report_path, None)
            print("INPUT_SHA256_FAIL")
            print(f"TF48_PART{PART_INDEX}_OUTPUT_ZIP={output}")
            raise WorkerFailure("INPUT_BLOCKED: corrected 16 T manifest verification failed")
        for name in (COIL_FILE, INPUT_FILE):
            if not (data_root / name).is_file():
                raise FileNotFoundError(data_root / name)
        report["direct_ripple"] = compute_direct_ripple(data_root)
        report["tf_edge_ripple_strict_pass"] = report["direct_ripple"]["strict_pass"]
        if not report["tf_edge_ripple_strict_pass"]:
            report["status"] = "TF_RIPPLE_REJECTED"
            report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
            output = make_output_zip(work, data_root, report_path, None)
            print("STATUS=TF_RIPPLE_REJECTED")
            print(f"TF48_PART{PART_INDEX}_OUTPUT_ZIP={output}")
            raise WorkerFailure("TF_RIPPLE_REJECTED")
        xgrid = find_xgrid(search_roots(work))
        native_dir = None
        if xgrid is None:
            report.update({"status": "NATIVE_XGRID_NOT_FOUND", "native_xgrid": None, "native_mgrid_present": False})
        else:
            print(f"TF48_PART{PART_INDEX}_XGRID_SOURCE={xgrid}")
            native_dir = Path(tempfile.mkdtemp(prefix=f"tf48-part{PART_INDEX}-native-", dir=str(work)))
            for name in (COIL_FILE, INPUT_FILE, f"extcur.tomahawk_tf48_part{PART_INDEX}"):
                shutil.copy2(data_root / name, native_dir / name)
            staged_xgrid, native_environment = stage_xgrid(xgrid, native_dir)
            log_path = native_dir / f"xgrid_part{PART_INDEX}.log"
            return_code, native_elapsed_s, timed_out = run_native_xgrid(
                staged_xgrid,
                native_dir / INPUT_FILE,
                native_dir,
                log_path,
                native_environment,
            )
            result_path = native_dir / EXPECTED_MGRID
            report.update({
                "native_xgrid": xgrid,
                "native_return_code": return_code,
                "native_elapsed_s": native_elapsed_s,
                "native_internal_timeout_enabled": nonnegative_env_seconds("TF48_XGRID_TIMEOUT_SECONDS", 0.0) > 0.0,
                "native_mgrid_present": result_path.is_file(),
                "native_log": str(log_path),
            })
            if timed_out:
                report["status"] = "NATIVE_XGRID_TIMEOUT"
            elif return_code == 0 and result_path.is_file():
                report["native_mgrid"] = inspect_mgrid(result_path)
                report["native_pass"] = bool(report["native_mgrid"]["valid"])
                report["status"] = "NATIVE_PART_PASS" if report["native_pass"] else "NATIVE_MGRID_INVALID"
            else:
                report["status"] = "NATIVE_XGRID_FAILED"
        report["accepted_for_assembly"] = bool(report.get("native_pass", False) and report["tf_edge_ripple_strict_pass"])
        report["accepted_for_downstream_solver"] = False
        report["elapsed_s"] = time.time() - started
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
        output = make_output_zip(work, data_root, report_path, native_dir)
        print(f"TF48_PART{PART_INDEX}_PHYSICAL_COILS=48")
        print(f"TF48_PART{PART_INDEX}_FIELD_PERIODS=16")
        print(f"TF48_PART{PART_INDEX}_DIRECT_EDGE_RIPPLE_PERCENT={report['direct_ripple']['edge_ripple_percent']:.10f}")
        print(f"TF48_PART{PART_INDEX}_TF_EDGE_RIPPLE_STRICT_PASS={report['tf_edge_ripple_strict_pass']}")
        print(f"TF48_PART{PART_INDEX}_STATUS={report['status']}")
        print(f"TF48_PART{PART_INDEX}_OUTPUT_ZIP={output}")
        if report["status"] != "NATIVE_PART_PASS":
            if native_dir is not None:
                print_log_tail(native_dir / f"xgrid_part{PART_INDEX}.log")
            raise WorkerFailure(f"TF48 Part {PART_INDEX} failed closed: {report['status']}")
    except WorkerFailure:
        raise
    except Exception as exc:
        report.update({"status": "RUNTIME_ERROR", "error": repr(exc), "elapsed_s": time.time() - started})
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
        output = make_output_zip(work, work, report_path, None)
        print(f"TF48_PART{PART_INDEX}_RUNTIME_ERROR={exc!r}")
        print(f"TF48_PART{PART_INDEX}_OUTPUT_ZIP={output}")
        raise


main()
